# GEC Toolkit — Pro Edition

Adds:
1) **Frontier Registry** (versioned Cmax per domain)
2) **Batch Processing** from CSV → results CSV
3) **Auto-Exports**: CSV summaries and PNG charts (investor-friendly)
4) Minimal narrative to explain *what the financier needs to see*


In [ ]:
import math
import json
from typing import Dict, Iterable, Optional, List
import matplotlib.pyplot as plt

def _safe_div(a: float, b: float) -> float:
    return float('nan') if b == 0 else a / b

def gec0(Y: float, X: float, Cmax: float) -> float:
    eta = _safe_div(Y, X)
    if math.isnan(eta) or Cmax <= 0:
        return float('nan')
    return eta / Cmax

def _geom_mean(values: Iterable[float], weights: Optional[Iterable[float]] = None) -> float:
    vals = list(values)
    if any(v <= 0 for v in vals):
        return float('nan')
    if weights is None:
        w = [1.0/len(vals)] * len(vals)
    else:
        w = list(weights)
        s = sum(w)
        if s <= 0:
            return float('nan')
        w = [wi/s for wi in w]
    log_sum = 0.0
    for v, wi in zip(vals, w):
        log_sum += wi * math.log(v)
    return math.exp(log_sum)

def csk_lambda(kappa: Dict[str, float], weights: Optional[Dict[str, float]] = None) -> float:
    keys = list(kappa.keys())
    vals = [max(1e-9, min(1.0, float(kappa[k]))) for k in keys]
    if weights is None:
        w = None
    else:
        w = [float(weights.get(k, 0.0)) for k in keys]
    return _geom_mean(vals, w)

def gec0_variance_delta(Y: float, X: float, Cmax: float, varY: float, varX: float, varC: float) -> float:
    if X <= 0 or Cmax <= 0:
        return float('nan')
    term1 = (1.0 / (Cmax * X))**2 * varY
    term2 = (Y / (Cmax * X**2))**2 * varX
    term3 = (Y / (X * Cmax**2))**2 * varC
    return term1 + term2 + term3

def frontier_audit_and_renorm(series: List[dict], Cnew: float) -> List[dict]:
    if Cnew <= 0:
        raise ValueError("Cnew must be > 0")
    out = []
    for rec in series:
        Y = float(rec.get("Y", float("nan")))
        X = float(rec.get("X", float("nan")))
        Cold = float(rec.get("Cold", float("nan")))
        g_new = gec0(Y, X, Cnew)
        adj = float("nan") if Cold <= 0 else (Cold / Cnew)
        flag = bool(g_new > 1.0) if not math.isnan(g_new) else False
        rec2 = dict(rec)
        rec2["GEC0_new"] = g_new
        rec2["adj_factor"] = adj
        rec2["flag_error"] = flag
        out.append(rec2)
    return out


## Frontier Registry
A simple JSON that records **domain → Cmax**, rationale, and versioning for transparency.
Edit in-place or load from an external file.

In [ ]:
# Example registry (edit/replace with your own)
frontier_registry = {
    "communications": {
        "Cmax": 3.46,
        "units": "bit/s/Hz at ~10 dB",
        "source": "Shannon capacity approximation (illustrative)",
        "version": "2025-10-25"
    },
    "thermodynamics_carnot": {
        "Cmax": 0.79,
        "units": "η_max (Carnot), illustrative",
        "source": "Carnot bound / exergy references",
        "version": "2025-10-25"
    },
    "ml_imagenet": {
        "Cmax": 0.905,
        "units": "Top-1 accuracy (SOTA proxy), illustrative",
        "source": "Public SOTA proxy (example)",
        "version": "2025-10-25"
    },
    "finance_roa": {
        "Cmax": 0.020,
        "units": "ROA (top-decile empirical)",
        "source": "Internal frontier proxy",
        "version": "2025-10-25"
    },
    "governance_tpr_at_fpr005": {
        "Cmax": 0.95,
        "units": "TPR @ FPR=0.05",
        "source": "Policy/operating target",
        "version": "2025-10-25"
    }
}

def save_frontier_registry(path="frontier_registry.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(frontier_registry, f, indent=2)

def load_frontier_registry(path="frontier_registry.json"):
    global frontier_registry
    with open(path, "r", encoding="utf-8") as f:
        frontier_registry = json.load(f)
    return frontier_registry

# Save a copy by default
save_frontier_registry("frontier_registry.json")
frontier_registry


## Batch Processing: CSV → Results CSV
Expected input columns:

- `domain` (key into frontier registry)
- `id` (row id)
- `Y`, `X`
- Optional secondary: `S,H,D,R,E` in [0,1]
- Optional variances: `varY,varX,varC` (if omitted, no error bars)


In [ ]:
import csv

def process_csv(input_csv: str, output_csv: str, registry_path: str = "frontier_registry.json"):
    reg = load_frontier_registry(registry_path)

    rows_in = []
    with open(input_csv, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows_in.append(r)

    out_rows = []
    for r in rows_in:
        domain = r.get("domain","").strip()
        Cmax = float(reg[domain]["Cmax"]) if domain in reg else float("nan")
        Y = float(r.get("Y", "nan"))
        X = float(r.get("X", "nan"))

        g0 = gec0(Y, X, Cmax)

        # secondary diagnostics (λ)
        S = float(r.get("S", 1.0) or 1.0)
        H = float(r.get("H", 1.0) or 1.0)
        D = float(r.get("D", 1.0) or 1.0)
        R2= float(r.get("R", 1.0) or 1.0)
        E = float(r.get("E", 1.0) or 1.0)
        lam = csk_lambda({"S":S,"H":H,"D":D,"R":R2,"E":E})

        # uncertainty if provided
        varY = r.get("varY")
        varX = r.get("varX")
        varC = r.get("varC")
        if varY and varX and varC:
            var = gec0_variance_delta(Y, X, Cmax, float(varY), float(varX), float(varC))
            sd = var**0.5
        else:
            var = float("nan")
            sd = float("nan")

        out_rows.append({
            "id": r.get("id",""),
            "domain": domain,
            "Y": Y, "X": X, "Cmax": Cmax,
            "GEC0": g0,
            "lambda": lam,
            "GEC0_times_lambda": (g0 * lam) if (g0==g0 and lam==lam) else float("nan"),
            "sd_GEC0": sd
        })

    # write output
    fieldnames = ["id","domain","Y","X","Cmax","GEC0","lambda","GEC0_times_lambda","sd_GEC0"]
    with open(output_csv, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in out_rows:
            w.writerow(r)

    return out_rows

def plot_gec0_summary(rows: List[dict], fig_path: str):
    labels = [r["id"] for r in rows]
    values = [r["GEC0"] for r in rows]
    import matplotlib.pyplot as plt
    plt.figure()
    plt.bar(range(len(labels)), values)
    plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
    plt.ylabel("GEC0")
    plt.title("Normalized Efficiency — Portfolio Summary")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200)
    plt.show()

def plot_composite_summary(rows: List[dict], fig_path: str):
    labels = [r["id"] for r in rows]
    values = [r["GEC0_times_lambda"] for r in rows]
    import matplotlib.pyplot as plt
    plt.figure()
    plt.bar(range(len(labels)), values)
    plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
    plt.ylabel("GEC0 · λ(κ)  [diagnostic]")
    plt.title("Composite Diagnostics — Portfolio Summary")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200)
    plt.show()


## Demo: Run & Export (Investor Pack)
This will:
1) Create a sample input CSV
2) Process it using the registry
3) Save a results CSV and two PNG charts


In [ ]:
import csv, os

sample_in = "sample_points.csv"
sample_out = "results_outputs.csv"

# 1) Create a small sample portfolio (edit freely)
with open(sample_in, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id","domain","Y","X","S","H","D","R","E"])
    w.writerow(["A1","communications",3.00,1.00,0.95,0.80,0.90,0.85,0.90])
    w.writerow(["B2","thermodynamics_carnot",0.3244,1.00,0.98,0.70,0.90,0.95,0.59])
    w.writerow(["C3","ml_imagenet",0.843,1.00,0.90,0.80,0.70,0.85,0.95])
    w.writerow(["D4","finance_roa",0.012,1.00,0.85,0.80,0.60,0.70,0.75])
    w.writerow(["E5","governance_tpr_at_fpr005",0.80,1.00,0.95,0.80,0.75,0.90,0.90])

# 2) Process
rows = process_csv(sample_in, sample_out)

# 3) Export charts
plot_gec0_summary(rows, "fig_gec0.png")
plot_composite_summary(rows, "fig_composite.png")

rows[:3]  # preview


## One-Paragraph Narrative for Financiers
We measure how close each asset/system is to its theoretical or empirical bound using **GEC₀** (normalized efficiency). A score of 1.0 means a system sits on the frontier; 0.8 means it achieves 80% of what physics/benchmarks say is possible. This is *unitless, comparable across domains,* and auditable: if the bound improves, we **renormalize** all historical scores with a transparent changelog. For diagnostics, we also show a composite `GEC₀·λ(κ)` using five operational factors (S,H,D,R,E) aggregated geometrically—this tells *why* two systems with the same GEC₀ differ. The investor pack (CSV + two charts) provides an at-a-glance portfolio summary and is reproducible from this notebook.